# **01 - Data Preparation**

This notebook covers:
1. Integration of metadata (XLSX) and texts (CSV)
2. Reconstruction of the multilabel structure
3. Text cleaning and normalization
4. ODS label encoding (one-hot encoding)
5. Dataset splitting (train / validation / test)

### **0. Imports & Configuration**

In [1]:
import pandas as pd
import numpy as np
import os
import re

import sys
import unicodedata

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

sys.path.append('../src')
from utils import RAW_METADATA_PATH, RAW_TEXT_CSV_DIR, OUTPUT_DIR, RANDOM_SEED, TEST_SIZE, VAL_SIZE, ODS_ALL
from schema import Description, Metadata

# Create output directory if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True) 

### **1. Load Data**

#### 1.1 Metadata (XLSX)

In [2]:
df_meta = pd.read_excel(RAW_METADATA_PATH, dtype=str)

print(f'Metadata loaded: {df_meta.shape}')
# df_meta.head()

# Check column names
print('Columns (metadata):', df_meta.columns.tolist())

# print('\nNull values per column:')
# print(df_meta.isnull().sum())

# print('\nUnique values per column:')
# print(df_meta.nunique())

Metadata loaded: (60114, 8)
Columns (metadata): ['ANH_ID', 'ANU_DATA_PUBLICACIO', 'ANU_NUM_REGISTRE', 'ORG_NOM', 'ANH_TITOL', 'Tipus Anunci', 'ODS_NOM', 'PDF']


#### 1.2 PDF Texts (CSV per year)

In [3]:
# Get CSV file paths
csv_files = []
for filename in os.listdir(RAW_TEXT_CSV_DIR):
    if filename.endswith('.csv'):
        csv_files.append(os.path.join(RAW_TEXT_CSV_DIR, filename))
print(f'CSV files found: {csv_files}')

CSV files found: ['../data/raw/description/anuncis_BOPB_2024_contingut.csv', '../data/raw/description/anuncis_BOPB_2023_contingut.csv', '../data/raw/description/anuncis_BOPB_2022_contingut.csv']


In [4]:
# Load each CSV and add year column
dfs_text = []
for f in sorted(csv_files):
    year = re.search(r'\d{4}', os.path.basename(f)).group()
    df_tmp = pd.read_csv(f, dtype=str)
    df_tmp[Description.YEAR] = year
    dfs_text.append(df_tmp)
    print(f'  {year}: {df_tmp.shape[0]} records')

df_texts = pd.concat(dfs_text, ignore_index=True)
print(f'Total texts: {df_texts.shape}')
# df_texts.head()

  2022: 35377 records
  2023: 30458 records
  2024: 10254 records
Total texts: (76089, 3)


In [5]:
print('Columns (description):', df_texts.columns.tolist())

Columns (description): ['id', 'text', 'any_publicacio']


### **2. Integration & Multilabel Reconstruction**

In [6]:
# Group all ODS labels per announcement into a deduplicated list
# Announcements with no ODS assigned are kept with an empty list
df_valid = df_meta.dropna(subset=[Metadata.ID_REGISTRE]).copy()
df_valid[Metadata.ODS] = df_valid[Metadata.ODS].str[:6].str.rstrip()  # trim to first 6 chars to match ODS_ALL format

df_labels = (
    df_valid
    .groupby(Metadata.ID_REGISTRE)[Metadata.ODS]
    .apply(lambda x: list(set(v for v in x if pd.notna(v))))
    .reset_index()
    .rename(columns={Metadata.ODS: Metadata.ODS_LIST})
)

# Merge with remaining metadata columns (deduplicated by ID)
meta_cols = [c for c in df_meta.columns if c != Metadata.ODS]
df_meta_unique = df_meta[meta_cols].drop_duplicates(subset=[Metadata.ID_REGISTRE])
df_labels = df_labels.merge(df_meta_unique, on=Metadata.ID_REGISTRE, how='left')

print(f'Unique announcements (including those with no ODS): {len(df_labels)}')
print(f'  - With at least one ODS: {df_labels[Metadata.ODS_LIST].apply(len).gt(0).sum()}')
print(f'  - With no ODS assigned:  {df_labels[Metadata.ODS_LIST].apply(len).eq(0).sum()}')
# df_labels.head()

Unique announcements (including those with no ODS): 35520
  - With at least one ODS: 19284
  - With no ODS assigned:  16236


In [7]:
# Merge text descriptions into the metadata dataframe
df = df_labels.merge(
    df_texts[[Description.ID_REGISTRE, Description.TEXT_RAW, Description.YEAR]].rename(columns={Description.ID_REGISTRE: Metadata.ID_REGISTRE}),
    on=Metadata.ID_REGISTRE,
    how='left'
)

print(f'Integrated dataset: {df.shape}')
print(f'Records without text: {df[Description.TEXT_RAW].isna().sum()}')
# df.head()

Integrated dataset: (35520, 10)
Records without text: 143


### **3. Text Cleaning & Normalization**

In [8]:
def clean_text(text: str) -> str:
    """Cleans and normalizes raw text for vectorization."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)                       # unify accented character variants (e.g. a + ´ → à)
    text = text.lower()                                             # homogenize casing
    text = re.sub(r'[\x00-\x1f\x7f]', ' ', text)                    # strip non-printable control characters
    text = re.sub(r'https?://\S+|www\.\S+', '', text)               # remove URLs
    text = re.sub(r'[^\w\s.,;:()/\-àáèéíïòóúüçñ·]', ' ', text)      # drop symbols irrelevant to the model
    text = re.sub(r'\s+', ' ', text).strip()                        # collapse multiple whitespace
    return text

In [9]:
# Apply clean_text to title and description
df[Metadata.TITLE_CLEAN] = df[Metadata.TITLE].apply(clean_text)
df[Description.TEXT_CLEAN] = df[Description.TEXT_RAW].apply(clean_text)

In [10]:
# Concatenate cleaned title and description into a single text field
df[Metadata.FULL_TEXT] = (
    df[Metadata.TITLE_CLEAN].fillna('') + ' ' +
    df[Description.TEXT_CLEAN].fillna('')
).str.strip()

# df.head()

### **4. Label Encoding (*MultiLabelBinarizer*)**

In [11]:
mlb = MultiLabelBinarizer(classes=ODS_ALL)
Y = mlb.fit_transform(df[Metadata.ODS_LIST])

df_labels_bin = pd.DataFrame(Y, columns=mlb.classes_, index=df.index)
df = pd.concat([df.reset_index(drop=True), df_labels_bin.reset_index(drop=True)], axis=1)

print('Binary label matrix:')
print(f'  Shape: {Y.shape}')
print(f'  Encoded ODS: {mlb.classes_.tolist()}')

Binary label matrix:
  Shape: (35520, 17)
  Encoded ODS: ['ODS 1', 'ODS 2', 'ODS 3', 'ODS 4', 'ODS 5', 'ODS 6', 'ODS 7', 'ODS 8', 'ODS 9', 'ODS 10', 'ODS 11', 'ODS 12', 'ODS 13', 'ODS 14', 'ODS 15', 'ODS 16', 'ODS 17']


### **5. Dataset Splitting**

In [12]:
# Multilabel stratified split — preserves ODS distribution across all partitions
X = df[Metadata.FULL_TEXT].values

# Train+val / test split
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_val_idx, test_idx = next(msss.split(X, Y))

X_train_val, Y_train_val = X[train_val_idx], Y[train_val_idx]
X_test,      Y_test      = X[test_idx],      Y[test_idx]

# Train / val split (adjusted size relative to train+val subset)
val_size_adj = VAL_SIZE / (1 - TEST_SIZE)
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_size_adj, random_state=RANDOM_SEED)
train_idx, val_idx = next(msss2.split(X_train_val, Y_train_val))

X_train, Y_train = X_train_val[train_idx], Y_train_val[train_idx]
X_val,   Y_val   = X_train_val[val_idx],   Y_train_val[val_idx]

print(f'Train:      {len(X_train):>5} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Validation: {len(X_val):>5} ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test:       {len(X_test):>5} ({len(X_test)/len(X)*100:.1f}%)')

Train:      24864 (70.0%)
Validation:  5328 (15.0%)
Test:        5328 (15.0%)


### **6. Export Processed Dataset**

In [13]:
# Save full clean dataset
df_out = df[[Metadata.ID_REGISTRE, Metadata.FULL_TEXT, Metadata.ODS_LIST] + ODS_ALL].copy()
df_out.to_parquet(os.path.join(OUTPUT_DIR, 'dataset_clean.parquet'), index=False)

# Save train / val / test splits
for name, idx_arr in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    # Reconstruct absolute indices
    if name == 'test':
        abs_idx = train_val_idx[np.arange(len(train_val_idx))]  # placeholder
        abs_idx = test_idx
    elif name == 'train':
        abs_idx = train_val_idx[train_idx]
    else:
        abs_idx = train_val_idx[val_idx]

    df_out.iloc[abs_idx if name != 'test' else np.arange(len(df_out))[test_idx]].to_parquet(
        os.path.join(OUTPUT_DIR, f'split_{name}.parquet'), index=False
    )

# Save binarizer for future use
import pickle
with open(os.path.join(OUTPUT_DIR, 'mlb.pkl'), 'wb') as f:
    pickle.dump(mlb, f)

print('Dataset and splits exported successfully.')

Dataset and splits exported successfully.
